# SCS 3546: Deep Learning
> Assignment 1: Deep Learning Using Keras

### Your name & student number:

<pre> Name: Miguel Morales </pre>

<pre> Student number: 466910. </pre>

## Assignment Description

In this assignment you will demonstrate your ability to:

- Train a neural network using Keras to solve a regression problem.

- Perform sensible data preprocessing.

- Experiment with hyperparemter tuning and different model architectures to achieve best performance.



### Grade Allocation

**15 points total**

- Part 1: 4 Marks
- Part 2: 9 Marks
- Clarity: 2 Marks

The marks for clarity are awarded for code documentation and how well you explained/supported your answers, including the use of visualizations where appropriate.

In [ ]:
# OPTIONAL : For model evaluation and for those who choose to use the KerasRegressor from keras.wrappers, you would need to install tensorflow 2.12 or 2.11.another option is to use the scikeras.
# please note that you may have many options and the below libraries are just meant to help you and to provide options
# !pip install tensorflow==2.12
# from keras.wrappers.scikit_learn import KerasRegressor

# !pip install scikeras
#from scikeras.wrappers import KerasClassifier, KerasRegressor

# Miguel's choice of TensorFlow version for my local environment
# TensorFlow 2.16.2 was installed since 2.12 is outdated
#%pip install tensorflow==2.16.2


In [ ]:
# setting up the notebook with important libraries
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
# Additional libraries used throughout the notebook
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split

# Versions differ between local and Colab; only Keras 3.x is required (E10)
print(f"Python       : {sys.version.split()[0]}")
print(f"TensorFlow   : {tf.__version__}")
print(f"Keras        : {keras.__version__}")
print(f"scikit-learn : {sklearn.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"pandas       : {pd.__version__}")

# My output for library versions in my local environment
# Python       : 3.12.7
# TensorFlow   : 2.16.2
# Keras        : 3.15.1
# scikit-learn : 1.5.1
# NumPy        : 1.26.4
# pandas       : 2.2.2

# Ensure Keras 3.x is used (E10) - it stops the notebook if an incompatible version is found
assert keras.__version__.startswith("3."), f"Keras 3.x required, found {keras.__version__}"

# Preamble

### Hyperparameters

A hyperparameter is a parameter whose value is set before the learning process begins.

Some important Neural Networks hyperparameters include:

- number of hidden layers
- number of neurons
- learning rate
- activation function
- optimizer settings

Hyperparameters are crucial to the performance, speed, and quality of the machine learning models.

Through Hyper parameter optimization, we find a tuple (best combination) of hyperparameters that yields an optimal model which minimizes a predefined loss function on given test data.

Important hyperparameters that could be tuned include:

- num_hidden_layers
- neurons_per_layer
- dropout_rate
- activation
- optimizer
- learning_rate
- batch_size

### Loss Function

- MSE (Mean Squared Error) is used as the score/loss function that will be minimized for hyperparameter optimization.
- In this assignment, we are going to use Cross-Validation to calculate the score (MSE) for a given set of hyperparameter values

- MSE is a desirable metric because by taking the square root gives us an error value we can directly understand in the context of the problem; for example, in this assignment it translates to thousands of dollars

- Note: Your results may vary given the stochastic nature of the algorithm, evaluation procedure, or differences in numerical precision

## Dataset

We will be using mock data for this assignment. the outcome is the sale price for some products. we have created 13 features to predict the sale price.

- import provided mock data into your notebook.

- You are **not** expected to perform Exploratory Data Analysis (EDA) on this dataset.

- For the purpose of this assignment, your model's performance is not an important factor by itself - that is how far your sale price prediction is. The important factor is the changes that you observe when changing model architectures.

- The information that follow are meant to be optional and to help you get familiar with the data. Your efforts on this assignment should focus on **model training and hyperparameter tuning**, not on EDA.


# Assignment Start
***

- Please follow all instructions carefully.

- Use MSE (Mean Squared Error) as the score/loss function that will be minimized during optimization.








In [ ]:
# Global constants: every experiment reads its settings from here
SEED = 42
EPOCHS = 100        # template Part 2 spec; Part 1 matches it (D05)
N_FOLDS = 3         # template: "e.g. with 3 cross-folds"
BATCH_SIZE = 32     # Keras default, same for every model (D10)
VAL_SIZE = 0.2      # Part 1 hold-out fraction (D04)

# Environment: the same notebook runs in Colab and locally
IN_COLAB = "google.colab" in sys.modules
# Path to the data folder in Colab; to adjust if folder structure is different
COLAB_DATA_DIR = Path("/content/drive/MyDrive/AI Certificate/DeepLearning/dl-assignment1/data")

## Helper Functions

This section contains reusable functions for seeding, locating the data, building and training models, and plotting. Each one is added here just before it is first used.

In [ ]:
def set_global_seed(seed=SEED):
    """Seed Python, NumPy and TensorFlow so runs are repeatable."""
    keras.utils.set_random_seed(seed)


def get_data_dir(candidates=None, marker_file="X_train.csv"):
    """Return the first candidate folder that contains `marker_file`; raise if none does."""
    if candidates is None:
        if IN_COLAB:
            candidates = [COLAB_DATA_DIR, Path("/content")]
        else:
            # The kernel may start in notebooks/ or in the repo root
            candidates = [Path("data"), Path("../data")]

    for folder in candidates:
        if (Path(folder) / marker_file).is_file():
            return Path(folder).resolve()

    tried = "\n  ".join(str(Path(folder).resolve()) for folder in candidates)
    raise FileNotFoundError(f"'{marker_file}' not found. Paths tried:\n  {tried}")

In [ ]:
# Functions for building, training, and evaluating a fully-connected regression model with Keras
# Starts from an input layer of 13 features, adds one hidden layer per entry in hidden_units,
# then a 1-unit output with no activation. It's compiled with MSE loss and tracks MAE.
def build_model(hidden_units, n_features, activation="relu", optimizer="adam", name=None):
    """Build and compile a fully-connected regression network (MSE loss, MAE metric)."""
    model = Sequential([keras.Input(shape=(n_features,))], name=name)
    for units in hidden_units:
        model.add(Dense(units, activation=activation))
    model.add(Dense(1))  # linear output: the target is an unbounded price
    model.compile(optimizer=optimizer, loss="mse", metrics=["mae"])
    return model

# Scale features using StandardScaler, fitting only on the training set
# Fits the scaler on X_fit only, then applies it to both arrays.
# This avoids "data leakage" by not using information from the validation set during scaling
def scale_features(X_fit, X_other):
    """Fit a StandardScaler on X_fit only, then transform both arrays."""
    scaler = StandardScaler().fit(X_fit)
    return scaler.transform(X_fit), scaler.transform(X_other)

# Train a model with an 80/20 split and return the model and its training history
# random_state=SEED gives all three Part 1 models the same split.
# Clearing before seeding gives each model the same starting conditions.
def train_with_validation(X, y, hidden_units, activation="relu", optimizer="adam", name=None):
    """Part 1: train a fresh model on a shuffled 80/20 split; return the model and its history."""
    X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=VAL_SIZE, random_state=SEED)
    X_tr, X_val = scale_features(X_tr, X_val)

    # Clear before seeding so the seed applies to the new model's weights
    keras.backend.clear_session()
    set_global_seed()
    model = build_model(hidden_units, X.shape[1], activation, optimizer, name)

    history = model.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                        epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)
    return model, history

# Compute final metrics from a Keras training history
# For reporting results in 1a and for a comparison table in 1c
# RMSE = sqrt(MSE) puts the loss back in $1000s, the same units as MAE so it is easier to interpret.
def final_metrics(history):
    """Final-epoch training and validation MSE, RMSE and MAE from a Keras history."""
    h = history.history
    return {"train_mse": h["loss"][-1], "val_mse": h["val_loss"][-1],
            "train_rmse": np.sqrt(h["loss"][-1]), "val_rmse": np.sqrt(h["val_loss"][-1]),
            "train_mae": h["mae"][-1], "val_mae": h["val_mae"][-1]}

# Plot learning curves for training and validation metrics
# Two panels side by side: MSE and MAE, training vs validation, log y-axis.
# Log y-axis option was added, to visualize when needed because MSE drops rapidly in the first epochs;
# thus, the log scale keeps the gap between training and validation visible.
def plot_learning_curves(history, title, log_scale_y_axis=True):
    """Plot training vs validation MSE and MAE per epoch (log scale so late epochs stay visible)."""
    h = history.history
    epochs = range(1, len(h["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metric, label in zip(axes, ["loss", "mae"], ["MSE (loss)", "MAE ($1000s)"]):
        ax.plot(epochs, h[metric], label="Training")
        ax.plot(epochs, h[f"val_{metric}"], label="Validation")
        ax.set(title=f"{title}: {label}", xlabel="Epoch", ylabel=f"{label}, log scale" if log_scale_y_axis else f"{label}")
        if log_scale_y_axis:
            ax.set_yscale("log")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.show()

#Data Import

The code below imports the data for you as dataframe, then you may need to convert it to numpy arrays.  

In [ ]:
# You can use the code below to import the data
# The code below was edited so it runs both in Colab and locally: Drive is mounted only in Colab,
# and the data folder is found by get_data_dir() instead of a hard-coded path.

# Option 3 - from local using the files that are available
import pandas as pd

# mount the colab
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')


# Next, You need to upload tables to your G drive -  then you may need to update  paths below
DATA_DIR = get_data_dir()
print(f"Reading data from: {DATA_DIR}")

# load the  data into a pandas dataframe for easy viewing and manipulation
# Real filenames use a capital X (Colab paths are case-sensitive);
# index_col=0 because each CSV starts with an unnamed row-index column.
df_train_y = pd.read_csv(DATA_DIR / "y_train.csv", index_col=0)
df_test_y = pd.read_csv(DATA_DIR / "y_test.csv", index_col=0)
df_train_x  =  pd.read_csv(DATA_DIR / "X_train.csv", index_col=0)
df_test_x =  pd.read_csv(DATA_DIR / "X_test.csv", index_col=0)

In [ ]:
# Hint : to make it easier, you may try to standardize your data upfront

In [ ]:
# Please note that I deliberately deviate from the hint above, meaning features are not standardized here.
# The scaler is fitted later on the training rows of each split only, so validation
# rows never influence the mean/std used to scale them (which I remember from previous course 
# that it avoids data leakage).

X_train = df_train_x.to_numpy()
X_test = df_test_x.to_numpy()
# The tables are flattened into 1-D numpy arrays for compatibility with Keras and sklearn
y_train = df_train_y.to_numpy().ravel()   # 1-D target, as Keras and sklearn expect
y_test = df_test_y.to_numpy().ravel()

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")
assert X_train.shape == (404, 13) and X_test.shape == (102, 13), "Unexpected feature shapes"
assert y_train.shape == (404,) and y_test.shape == (102,), "Unexpected target shapes"

In [ ]:
# These are quick sanity checks (not EDA): 
# Just checking for no missing values, and for the target's scale,
# which is needed to judge whether an MAE is small or large (units: thousands of dollars).
print("Missing values:", int(df_train_x.isna().sum().sum() + df_test_x.isna().sum().sum()))
print(pd.Series(y_train, name="y_train").describe().round(2))

# Part 1: Impact of Changing Model Architecture

In this section, we will be comparing a simple single-layer baseline model with two other models having a different network topology.

## a) Baseline model [2 points]

Use Keras to develop a baseline neural network model that has **one single fully-connected hidden layer with the same number of neurons as input features (i.e. 13 neurons).**

Make sure to **standardize** your features (i.e. subtract mean and divide by standard deviation) before training your model. You can also perform any other data-preprocessing that you deem necessary.

- Note: No activation function is used for the output layer because it is a regression problem and we are interested in predicting numerical values directly without transformation.

- The ADAM optimization algorithm should be used to optimize mean squared error loss function.

- Plot learning curves and report on both training and validation performance.

In [ ]:
# your code here, use as many cells as you need

# Baseline: one hidden layer with 13 ReLU units (one per input feature), Adam, MSE loss.
# Features are standardized inside train_with_validation, using the 80% training rows only.
model_1a, history_1a = train_with_validation(X_train, y_train, hidden_units=[13], name="baseline_13")
model_1a.summary()

In [ ]:
plot_learning_curves(history_1a, "1a Baseline (13)", log_scale_y_axis=False)
plot_learning_curves(history_1a, "1a Baseline (13)")

# Collected across 1a-1c for the Part 1 comparison
part1_results = {"1a: 13": final_metrics(history_1a)}
pd.DataFrame(part1_results).T.round(2)

#### Naive Baseline model: 
A naive baseline model was added. It always predicts the mean of the training split. This provides a reference point for evaluating the performance of more complex models. A useful model should be able to outperform this baseline.

In [ ]:
# Naive baseline for context: always predict the training split's mean price; a useful model should beat it.
# Same split call as train_with_validation, so these are the same 81 validation rows used for 1a.
_, _, y_tr_naive, y_val_naive = train_test_split(X_train, y_train, test_size=VAL_SIZE, random_state=SEED)
naive_prediction = y_tr_naive.mean()

train_err = y_tr_naive - naive_prediction
val_err = y_val_naive - naive_prediction
naive_metrics = {"train_mse": np.mean(train_err ** 2), "val_mse": np.mean(val_err ** 2),
                 "train_rmse": np.sqrt(np.mean(train_err ** 2)), "val_rmse": np.sqrt(np.mean(val_err ** 2)),
                 "train_mae": np.mean(np.abs(train_err)), "val_mae": np.mean(np.abs(val_err))}

pd.DataFrame({"naive: mean": naive_metrics, "1a: 13": part1_results["1a: 13"]}).T.round(2)

### 1a --- Results and Observations ---
baseline: 1 hidden layer, 13 units, ReLU

|  | train MSE | val MSE | train RMSE | val RMSE | train MAE | val MAE |
| --- | --- | --- | --- | --- | --- | --- |
| naive (mean) | 91.81 | 57.07 | 9.58 | 7.55 | 7.01 | 5.70 |
| 1a: 13 | 39.86 | 29.62 | 6.31 | 5.44 | 4.22 | 4.24 |

*Final-epoch values (epoch 100). MSE is in ($1000s)²; RMSE and MAE are in $1000s.* 
*The model has 196 trainable parameters (the extra 394 are Adam's optimizer state, not model weights)*

I can summarize my analysis by 4 questions:

**The first question I asked myself was if it was still learning?** On the linear-scale plot, the curves looked like they flattened out, and I would have called it converged. But, with the log-scale plots I could see that both training and validation losses are still going down steadily at epoch 100. So we can conclude that with Adam's default learning rate, 100 epochs isn't enough for this small network. I kept 100 epochs anyway so all models are compared under the same protocol (the template specifies 100 epochs for Part 2, so Part 1 matches it).

**Overfitting or underfitting?** Underfitting. There's no widening gap between training and validation, and both errors are still falling, so the model has capacity it hasn't used yet.

**Why is validation error lower than training error?** This surprised me at first. Two things explain it:

-   Keras reports training loss as an average over the epoch while the weights are still improving, but validation is measured once at the end of the epoch with the latest weights.
-   More importantly, the validation split is simply easier. The naive baseline, which learns nothing, shows the same pattern (MSE 91.81 train vs 57.07 val). Its RMSE is roughly the spread of each split: 9.58 for training vs 7.55 for validation. So the gap comes from which 81 houses ended up in validation, not from the model. A single small split is noisy, which is one of the reasons Part 2 uses cross-validation.

The MAEs are almost equal (4.22 vs 4.24) while the MSEs differ, because squaring makes a few large errors (e.g. expensive houses) count much more in MSE than in MAE.

**How good is it?** A validation MAE of 4.24 means predictions are off by about $4,240 on average, around 19% of the mean price ($22.4k). RMSE is about $1,200 higher than MAE, so a few predictions are notably far off. Compared with the naive baseline, the network cuts validation error by roughly a quarter (MAE 5.70 → 4.24, RMSE 7.55 → 5.44). That's a real improvement, but a modest one, which fits a model that hasn't finished learning.

**Overall conclusion:** The model is underfitting, as indicated by the steadily decreasing training and validation losses and the lack of a widening gap between them. While it improves over the naive baseline, it hasn't fully utilized its capacity, suggesting that more training epochs or a different learning rate might yield better performance.

## b) Deeper Network [1 point]

Construct and evaluate a model with 2 dense layers having a smaller number of neurons (e.g. 16, 8).

In [ ]:
# your code here, use as many cells as you need

# Deeper network: two hidden ReLU layers (16 -> 8); same split, seed, Adam and epochs as 1a.
# 1a -> 1b changes both depth (1 -> 2 layers) and width (13 -> 16/8), so it is not a one-variable comparison.
model_1b, history_1b = train_with_validation(X_train, y_train, hidden_units=[16, 8], name="deeper_16_8")
model_1b.summary()

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

In [ ]:
plot_learning_curves(history_1b, "1b Deeper (16, 8)")

part1_results["1b: 16-8"] = final_metrics(history_1b)
pd.DataFrame({"naive: mean": naive_metrics, **part1_results}).T.round(2)

### 1b --- Results and Observations ---
deeper: 2 hidden layers, 16 → 8 units, ReLU

|  | train MSE | val MSE | train RMSE | val RMSE | train MAE | val MAE |
| --- | --- | --- | --- | --- | --- | --- |
| naive (mean) | 91.81 | 57.07 | 9.58 | 7.55 | 7.01 | 5.70 |
| 1a: 13 | 39.86 | 29.62 | 6.31 | 5.44 | 4.22 | 4.24 |
| 1b: 16-8 | 18.30 | 17.07 | 4.28 | 4.13 | 3.09 | 3.27 |

*Final-epoch values (epoch 100). Same split, seed, optimizer and number of epochs as 1a. The model has 369 trainable parameters, about 1.9× the 196 in 1a.*

Connecting this to some of the concepts we learned in the course, and comparing it against 1a, I can see the next 3 important observations:

**1. A different learning curve that converges faster.** The first thing I noticed is that the curves look quite different from 1a. There's still a slow start, but then a steep drop between roughly epochs 15 and 40, followed by a plateau. On the log scale the curves become almost flat (asymptotic) after about epoch 45, with only a slight downward slope left at epoch 100. So unlike 1a, which was still sliding down at the end, this will indicate to me that 1b has largely converged within the 100 epochs. The deeper network appears to learn much faster, although with a single run and one seed I can't say that for certain.

**2. Training finally drops below validation.** Around epoch 60 the training MAE crosses below the validation MAE for the first time, and stays there while the validation curve flattens. This is what I'd expect from a model that is now fitting the data well: it fits the rows it trains on better than unseen rows. It's also the first sign of overfitting. It isn't a problem yet, though: validation error isn't rising, and the gap is small (MAE 3.09 vs 3.27). What makes the crossover meaningful is that the naive baseline showed the validation split is the *easier* one, and the model is still doing better on training. In 1a this didn't happen, partly because the loss was still dropping fast, which makes the epoch-averaged training loss look worse.

**3. Depth or width?** I can't tell from this experiment. Going from 1a to 1b changed three things at once: the number of layers (1 → 2), the number of hidden units (13 → 24) and the number of parameters (196 → 369). The improvement could come from any of these, or from a combination. 1c keeps two layers and only widens them, so it isolates the effect of width.

**Overall conclusions:** Validation MAE is now 3.27, meaning predictions are off by about $3,270 on average, around 15% of the mean price (down from about 19% in 1a)! Compared with the naive baseline, the error per house is about $2,400 lower (≈43% better MAE, ≈45% better RMSE), and compared with 1a it's about $1,000 lower (≈23% better)! The gap between RMSE and MAE also shrank, so the model makes fewer large misses, not just a smaller average error. Overall, this architecture clearly fits the data better and learns faster than 1a, but we can't attribute the improvements to the width or depth alone since we modified both parameters at once.

## c) Wider Network [1 point]

Construct and evaluate a wider model with more neurons (e.g. 32, 16).

In [ ]:
# your code here, use as many cells as you need

# Wider network: same two-layer shape as 1b, but twice the units per layer (32 -> 16).
# 1b -> 1c changes only the width, so this comparison isolates the effect of more neurons.
model_1c, history_1c = train_with_validation(X_train, y_train, hidden_units=[32, 16], name="wider_32_16")
model_1c.summary()

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

In [ ]:
plot_learning_curves(history_1c, "1c Wider (32, 16)")

part1_results["1c: 32-16"] = final_metrics(history_1c)
pd.DataFrame({"naive: mean": naive_metrics, **part1_results}).T.round(2)

### 1c --- Results and Observations ---
wider: 2 hidden layers, 32 → 16 units, ReLU

|  | params | train MSE | val MSE | train RMSE | val RMSE | train MAE | val MAE |
| --- | --- | --- | --- | --- | --- | --- | --- |
| naive (mean) | 0 | 91.81 | 57.07 | 9.58 | 7.55 | 7.01 | 5.70 |
| 1a: 13 | 196 | 39.86 | 29.62 | 6.31 | 5.44 | 4.22 | 4.24 |
| 1b: 16-8 | 369 | 18.30 | 17.07 | 4.28 | 4.13 | 3.09 | 3.27 |
| 1c: 32-16 | 993 | 10.65 | 9.53 | 3.26 | 3.09 | 2.33 | 2.26 |

*Final-epoch values (epoch 100). Same split, seed, optimizer and number of epochs as 1a and 1b. Only the width changed compared with 1b.*

I can group my conclusions around 3 main points:

**1. Width and total parameters matter:** Unlike 1a → 1b, this time I changed only one thing: same two layers, double the units per layer. Validation MAE dropped from 3.27 to 2.26, which is about 31% lower. So width (and with it the number of parameters, 369 → 993) clearly matters here. What we wouldn't be able to conclude from this experiment is how much the second layer contributed in 1b, because that comparison changed depth and width together. An idea for a fairer test of depth would keep the total number of units or parameters roughly the same and only vary the number of layers.

**2. There was even faster learning.** In the overlay plot, 1c reaches the naive baseline level around epoch 20, compared with about epoch 32 for 1b and about epoch 73 for 1a. My interpretation is that with more units, each update adjusts more weights, and the randomly initialized network starts with a richer set of features to build on. With a single seed I'd treat this as a likely explanation rather than a proven one.

**3. It seems the network is still learning at epoch 100.** On the log scale, both curves are still sloping down at the end, so 1c hasn't converged. More epochs would probably improve it further. Interestingly, and unlike 1b, the training error never crosses below validation (MAE 2.33 vs 2.26). I first expected the bigger model to overfit sooner, but it's the opposite here: 1c is still improving on unseen data, while 1b had flattened out and was starting to fit its training rows better than the validation rows. So at 100 epochs, 1c is neither overfitting nor done learning.

**Overall conclusion:** A validation MAE of 2.26 means predictions are off by about $2,260 on average, around 10% of the mean price (down from about 15% in 1b and 19% in 1a). Compared with the naive baseline, that's roughly $3,400 less error per house (≈60% lower MAE and ≈59% lower RMSE).

## Part 1 Comparison

All three models share the same 80/20 split, seed, optimizer (Adam), batch size and 100 epochs; only the architecture changes.

In [ ]:
part1_models = {"1a: 13": (model_1a, history_1a),
                "1b: 16-8": (model_1b, history_1b),
                "1c: 32-16": (model_1c, history_1c)}

# Validation MAE per epoch for all three architectures, with the naive baseline as reference
fig, ax = plt.subplots(figsize=(9, 4))
for name, (_, history) in part1_models.items():
    ax.plot(range(1, EPOCHS + 1), history.history["val_mae"], label=name)
ax.axhline(naive_metrics["val_mae"], color="gray", linestyle="--", label="naive: mean")
ax.set(title="Part 1: validation MAE by architecture", xlabel="Epoch",
       ylabel="Validation MAE ($1000s), log scale", yscale="log")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

part1_table = pd.DataFrame(part1_results).T
part1_table.insert(0, "params", [part1_models[name][0].count_params() for name in part1_table.index])
part1_table.round(2)

### Part 1 --- Additional Conclusions ---

Each step up in capacity improved results and made learning faster: 1a → 1b → 1c cut validation MAE from 4.24 to 3.27 to 2.26. Width clearly helped; the effect of depth on its own remains open. All three results come from a single split of 81 validation rows and one seed, so the exact numbers are noisy. That's why Part 2 uses cross-validation.

One other reflection that I have is if would bigger always be better? Probably not. The model trains on only 323 rows and already has about 3 parameters per training row. It isn't overfitting yet, likely because 100 epochs is a short budget. With much larger networks I'd expect training error to keep falling while validation error levels off and eventually rises, with more variation between splits.

In addition, as we discussed in the course larger models can bring a host of trade-offs. They often require more computation, which means longer training times and higher energy costs. They can also overfit more easily, (especially if the dataset is relatively small compared to the number of parameters). Also from what I read, larger models can be more complex to deploy and maintain. So while increasing model capacity can improve performance, it’s crucial to balance that with efficiency and generalization, and this is when careful design and experimantion, and tuning hyperparameters becomes important!



# Part 2: Hyperparameter Tuning Experiments

In the following experiments, you will evaluate and compare models trained with different hyperparameters. Please follow the specifications given for each model.

## a) Model 1 [2 points]

- 2 Dense layers:
  - The first with 64 neurons using a ReLU activation function.
  - The second with 64 neurons using a ReLU activation function.
- Choose an appropriate output layer and activation.
- Train model with 100 epochs and obtain cross-validated performance (e.g. with 3 cross-folds).
- Plot both loss and mean absolute error (i.e. learning curves) for both training and validation.
- Report MAE from CV with standard deviation.

In [ ]:
# your code here, use as many cells as you need

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

## b) Model 2 [2 points]

- 2 Dense layers:
  - The first with 128 neurons using a ReLU activation function.
  - The second with 64 neurons using a ReLU activation function.
- Choose an appropriate output layer and activation.
- Train model with 100 epochs and obtain cross-validated performance (e.g. with 3 cross-folds).
- Plot both loss and mean absolute error (i.e. learning curves) for both training and validation.
- Report MAE from CV with standard deviation.

In [ ]:
# your code here, use as many cells as you need

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

## c) Model 3 [2 points]

- Same as Model 2, but use tanh activation functions instead of relu.

In [ ]:
# your code here, use as many cells as you need

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

## d) Model 4 [2 points]

- Same as Model 2, but use the rmsprop optimizer when training.

In [ ]:
# your code here, use as many cells as you need

# be sure to comment on your results for each and every question, describing not just
# the what, but the 'how' and 'why' where possible, to demonstrate your understanding of
# course content

## e) Model Comparison [1 point]

Which model performed best? Offer your thoughts on why the particular choice of hyperparameters led to improved performance for this model.

In [ ]:
# explain WHY you think the best model was better than the rest, in terms
# of how those hyperparameters theoretically impact the model

# provide visualizations (e.g. tables or comparison plot) to support your response where possible

NOTE: 2 additional points are awarded based on code documentation and overall clarity of work.

In [ ]:
# We are looking for a clear explanation of results with each response. We want you to attempt to
# explain the _how_ and _why_ behind your answers, and not just the what, do demonstrate
# your knowledge of the concepts discussed in class. Answers should be backed up with
# visualizations (e.g. plots, charts).

# Code should be easy to follow by using sensical naming conventions for function and variable
# names, providing useful code comments, and refactoring repeated code into re-usable functions.